In [ ]:
%pip install -U yfinance pandas beautifulsoup4 requests
%pip install "transformers==4.44.2" "torch==2.5.1"

In [20]:
import yfinance as yf
import pandas as pd
from pprint import pprint
import requests
from bs4 import BeautifulSoup
import torch
import transformers
from transformers import pipeline
from transformers.utils import is_torch_available

In [21]:
search = yf.Search("AAPL") # Default gives us 8 articles
news = search.news

pprint(news[0])

df = pd.DataFrame(news)
display(df)

{'link': 'https://finance.yahoo.com/markets/stocks/articles/prediction-alphabet-larger-company-apple-232900856.html',
 'providerPublishTime': 1792193340,
 'publisher': 'Motley Fool',
 'relatedTickers': ['GOOG', 'AAPL', 'NVDA'],
 'thumbnail': {'resolutions': [{'height': 738,
                                'tag': 'original',
                                'url': 'https://media.zenfs.com/en/motleyfool.com/2a198da898e1e63c86ca12813bd2ad3a.jpg',
                                'width': 1400},
                               {'height': 140,
                                'tag': '140x140',
                                'url': 'https://s.yimg.com/lo/mysterio/api/5585041C3E4A9AE48999B8DB1B4500B8496044DF9A1E83D6972D50082B2BDB47/subgraphmysterio/resizefill_w140_h140/https:%2F%2Fmedia.zenfs.com%2Fen%2Fmotleyfool.com%2F2a198da898e1e63c86ca12813bd2ad3a.jpg',
                                'width': 140}]},
 'title': 'Prediction: Alphabet Will Be a Larger Company Than Apple by the End '
         

,uuid,title,publisher,link,providerPublishTime,type,thumbnail,relatedTickers
0,5e0035d1-847e-3a79-aa14-fedb45395bf8,Prediction: Alphabet Will Be a Larger Company ...,Motley Fool,https://finance.yahoo.com/markets/stocks/artic...,1792193340,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[GOOG, AAPL, NVDA]"
1,12a19546-b691-38fd-8a92-0a88758f06ef,Market Chatter: Apple Appoints Former Meta Exe...,MT Newswires,https://finance.yahoo.com/markets/stocks/artic...,1792191371,STORY,NaN,"[AAPL, META]"
2,596af860-67ee-3c68-a208-bfcfadf6a215,Taiwan Semiconductor Posts Record Q3 Results: ...,Zacks,https://finance.yahoo.com/markets/stocks/artic...,1792190160,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[TSM, TSM, AAPL, AMD, NVDA]"
3,c7e028b0-c8b5-3c35-8f5a-5a3b70d23374,Nasdaq 100 Records Worst Week In 18 Months As ...,Stocktwits,https://finance.yahoo.com/markets/stocks/artic...,1792188934,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[^GSPC, ^NDX, ^DJI, CRWV, ^RUT, SPY, DIA, IVZ,..."
4,4bb318f2-4454-3ac1-90a4-ce812b615d44,"Stock Market Today, Oct. 16: Apple Edges Highe...",Motley Fool,https://finance.yahoo.com/markets/stocks/artic...,1792187374,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[AAPL, MSFT, GOOG, ^IXIC, ^GSPC]"
5,bd9f3ace-7f96-34b1-b613-16c089d8198c,Tech Stocks Zig—and Apple Zags,The Wall Street Journal,https://finance.yahoo.com/m/bd9f3ace-7f96-34b1...,1792185614,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[AAPL, NVDA]"
6,5ee6712d-21f3-3b4b-aea1-531b69d766b5,This is the Cheapest Mag 7 Stock Right Now,Zacks,https://finance.yahoo.com/markets/stocks/artic...,1792184820,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[AMZN, TSLA, AAPL, BRK-B, GOOG]"
7,990d1896-43cd-3349-9be5-9dc32bf5f450,Apple's Next Legal Risk Could Reach $40 Billion,GuruFocus.com,https://finance.yahoo.com/technology/articles/...,1792183989,STORY,NaN,[AAPL]


In [22]:
article = news[0]
content = article.get("content", article)

url = (content.get("canonicalUrl") or {}).get("url") or content.get("link")

print("URL:", url)

# Download article HTML
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=15
)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Locate main article
article_html = soup.find("article")

# Extract paragraphs
if article_html:
    paragraphs = article_html.find_all("p")
    full_text = "\n\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
    )
else:
    full_text = None

# Display results
print("\nTITLE:")
print(content.get("title"))

print("\nFULL ARTICLE:")
print(full_text)

URL: https://finance.yahoo.com/markets/stocks/articles/prediction-alphabet-larger-company-apple-232900856.html

TITLE:
Prediction: Alphabet Will Be a Larger Company Than Apple by the End of 2027

FULL ARTICLE:
The above button links to Coinbase. Yahoo Finance is not a broker-dealer or investment adviser and does not offer securities or cryptocurrencies for sale or facilitate trading. Coinbase pays us for certain activity generated through this link. Prices displayed are informational.

Right now, Apple (NASDAQ: AAPL) and Alphabet (NASDAQ: GOOG) (NASDAQ: GOOGL) are the world's second- and third-largest companies, respectively. Apple's market cap sits just under $5 billion, while Alphabet's is $4.2 billion. While that's not an insignificant gap, I think Alphabet could close it over the next year and claim the title of the world's second-largest company.

It won't be easy, but I think there are two fronts where Alphabet can make up ground.

Missed Nvidia in 2009? This Rare Signal Is Flash

In [23]:
# Sentiment Analysis Experiment (Unstructured -> Structured Data)

import os
os.environ["USE_TF"] = "0"

import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline

sentiment = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    framework="pt",
    device="cpu"
)

if full_text and full_text.strip():

    results = sentiment(
        full_text,
        truncation=True,
        top_k=None
    )

    for result in results:
        print(result["label"], result["score"])

    df_sentiment = pd.DataFrame(results)

    display(df_sentiment)

else:
    print("No article text available.")

/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


neutral 0.8835909962654114
positive 0.08149680495262146
negative 0.034912239760160446


,label,score
0,neutral,0.883591
1,positive,0.081497
2,negative,0.034912
